# ResolveOne — Bronze, Silver and Gold Pipeline

## Layer rules

- **Bronze:** Read source values as text and add audit columns. Do not modify source values.
- **Silver:** Clean, type, enrich and validate every transaction.
- **Gold:** Keep one safe, agent-ready record per failed transaction.
- **Gate:** Gold is written only when the Silver chunk passes its quality tests.

In [ ]:
%pip install -q pyarrow

In [1]:
from __future__ import annotations

import hashlib
import json
import zipfile
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


# Repository paths
CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

ZIP_FILES = list(RAW_DIR.glob("*.zip"))

if not ZIP_FILES:
    raise FileNotFoundError(f"No ZIP file found inside {RAW_DIR}")

ZIP_PATH = ZIP_FILES[0]

# Pipeline configuration
CHUNK_SIZE = 500_000
RUN_ID = pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%SZ")
INGESTED_AT_UTC = pd.Timestamp.now(tz="UTC").isoformat()

SILVER_PATH = PROCESSED_DIR / "silver_transactions.parquet"
GOLD_PATH = PROCESSED_DIR / "gold_exception_cases.parquet"
QUALITY_PATH = PROCESSED_DIR / "quality_results.json"

# Prevent duplicate appends when the notebook is rerun
for output_path in [SILVER_PATH, GOLD_PATH, QUALITY_PATH]:
    output_path.unlink(missing_ok=True)

print("Project root:", PROJECT_ROOT)
print("Source ZIP:", ZIP_PATH)
print("Pipeline run ID:", RUN_ID)
print("Chunk size:", f"{CHUNK_SIZE:,}")

Project root: /home/labuser/Desktop/Persistent_Folder/Capstone/Resolve1
Source ZIP: /home/labuser/Desktop/Persistent_Folder/Capstone/Resolve1/data/raw/Financial Transactions Dataset Analytics.zip
Pipeline run ID: 20260806T132509Z
Chunk size: 500,000


In [2]:
with zipfile.ZipFile(ZIP_PATH, "r") as zip_file:
    files_in_zip = zip_file.namelist()


def find_file(filename: str) -> str | None:
    matches = [
        file_path
        for file_path in files_in_zip
        if Path(file_path).name.lower() == filename.lower()
    ]

    return matches[0] if matches else None


dataset_files = {
    "transactions": find_file("transactions_data.csv"),
    "cards": find_file("cards_data.csv"),
    "users": find_file("users_data.csv"),
    "mcc_codes": find_file("mcc_codes.json"),
    "fraud_labels": find_file("train_fraud_labels.json"),
}

missing_files = [
    name
    for name, path in dataset_files.items()
    if path is None
]

if missing_files:
    raise FileNotFoundError(
        f"Required files not found in ZIP: {missing_files}"
    )

dataset_files

{'transactions': 'transactions_data.csv',
 'cards': 'cards_data.csv',
 'users': 'users_data.csv',
 'mcc_codes': 'mcc_codes.json',
 'fraud_labels': 'train_fraud_labels.json'}

In [3]:
def clean_text(series: pd.Series) -> pd.Series:
    """Trim text and convert empty strings to missing values."""
    cleaned = series.astype("string").str.strip()
    return cleaned.mask(cleaned.eq(""))


def clean_money(series: pd.Series) -> pd.Series:
    """Convert values such as '$1,234.50' into numeric values."""
    cleaned = (
        clean_text(series)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
    )

    return pd.to_numeric(
        cleaned,
        errors="coerce",
    ).astype("Float64")


def clean_integer(series: pd.Series) -> pd.Series:
    """Convert a series into nullable integers."""
    return pd.to_numeric(
        clean_text(series),
        errors="coerce",
    ).astype("Int64")


def clean_yes_no(series: pd.Series) -> pd.Series:
    """Convert YES/NO and Yes/No values into nullable booleans."""
    return (
        clean_text(series)
        .str.upper()
        .map(
            {
                "YES": True,
                "NO": False,
            }
        )
        .astype("boolean")
    )


def mask_identifier(
    series: pd.Series,
    prefix: str,
) -> pd.Series:
    """Create deterministic masked identifiers."""

    def hash_value(value):
        if pd.isna(value):
            return pd.NA

        digest = hashlib.sha256(
            str(value).encode("utf-8")
        ).hexdigest()[:12]

        return f"{prefix}-{digest}"

    return series.map(hash_value).astype("string")

In [4]:
with zipfile.ZipFile(ZIP_PATH, "r") as zip_file:

    with zip_file.open(dataset_files["cards"]) as file:
        cards_raw = pd.read_csv(
            file,
            dtype=str,
            keep_default_na=False,
        )

    with zip_file.open(dataset_files["users"]) as file:
        users_raw = pd.read_csv(
            file,
            dtype=str,
            keep_default_na=False,
        )

    with zip_file.open(dataset_files["mcc_codes"]) as file:
        mcc_codes = json.load(file)

    with zip_file.open(dataset_files["fraud_labels"]) as file:
        fraud_labels = json.load(file)

fraud_map = fraud_labels["target"]


cards_safe = pd.DataFrame(
    {
        "card_id": clean_integer(cards_raw["id"]),
        "card_owner_client_id": clean_integer(
            cards_raw["client_id"]
        ),
        "card_brand": clean_text(cards_raw["card_brand"]),
        "card_type": clean_text(cards_raw["card_type"]),
        "has_chip": clean_yes_no(cards_raw["has_chip"]),
        "credit_limit": clean_money(cards_raw["credit_limit"]),
        "card_on_dark_web": clean_yes_no(
            cards_raw["card_on_dark_web"]
        ),
        "_card_joined": True,
    }
)


users_safe = pd.DataFrame(
    {
        "client_id": clean_integer(users_raw["id"]),
        "current_age": clean_integer(users_raw["current_age"]),
        "yearly_income": clean_money(
            users_raw["yearly_income"]
        ),
        "total_debt": clean_money(users_raw["total_debt"]),
        "credit_score": clean_integer(
            users_raw["credit_score"]
        ),
        "num_credit_cards": clean_integer(
            users_raw["num_credit_cards"]
        ),
        "_user_joined": True,
    }
)


assert cards_safe["card_id"].notna().all()
assert users_safe["client_id"].notna().all()

assert not cards_safe["card_id"].duplicated().any()
assert not users_safe["client_id"].duplicated().any()


print("Cards loaded:", f"{len(cards_safe):,}")
print("Users loaded:", f"{len(users_safe):,}")
print("MCC descriptions:", f"{len(mcc_codes):,}")
print("Fraud labels:", f"{len(fraud_map):,}")

# Remove raw sensitive tables from notebook memory
del cards_raw
del users_raw
del fraud_labels

Cards loaded: 6,146
Users loaded: 2,000
MCC descriptions: 109
Fraud labels: 8,914,963


In [5]:
SILVER_COLUMNS = [
    "transaction_id",
    "transaction_timestamp",
    "client_id",
    "card_id",
    "amount",
    "transaction_channel",
    "merchant_id",
    "merchant_city",
    "merchant_state",
    "merchant_zip",
    "mcc",
    "merchant_category",
    "errors",
    "fraud_label",
    "card_brand",
    "card_type",
    "has_chip",
    "credit_limit",
    "card_on_dark_web",
    "current_age",
    "yearly_income",
    "total_debt",
    "credit_score",
    "num_credit_cards",
    "_source_file",
    "_ingested_at_utc",
    "_bronze_row_num",
    "_pipeline_run_id",
]


GOLD_COLUMNS = [
    "exception_id",
    "transaction_id",
    "transaction_timestamp",
    "masked_client_id",
    "masked_card_id",
    "amount",
    "transaction_channel",
    "merchant_id",
    "merchant_city",
    "merchant_state",
    "merchant_zip",
    "mcc",
    "merchant_category",
    "error_types",
    "is_multi_error",
    "fraud_label",
    "card_brand",
    "card_type",
    "has_chip",
    "credit_limit",
    "card_on_dark_web",
    "current_age",
    "yearly_income",
    "total_debt",
    "credit_score",
    "num_credit_cards",
    "_source_file",
    "_ingested_at_utc",
    "_bronze_row_num",
    "_pipeline_run_id",
]

In [6]:
def build_bronze(
    raw_chunk: pd.DataFrame,
    row_offset: int,
) -> pd.DataFrame:
    """
    Keep source values as text and add audit columns.
    No source cleaning occurs in Bronze.
    """
    bronze = raw_chunk.copy()

    for column in bronze.columns:
        bronze[column] = bronze[column].astype("string")

    bronze["_source_file"] = dataset_files["transactions"]
    bronze["_ingested_at_utc"] = INGESTED_AT_UTC

    bronze["_bronze_row_num"] = pd.Series(
        range(
            row_offset + 1,
            row_offset + len(bronze) + 1,
        ),
        index=bronze.index,
        dtype="Int64",
    )

    bronze["_pipeline_run_id"] = RUN_ID

    return bronze


def build_silver(
    bronze: pd.DataFrame,
) -> tuple[pd.DataFrame, dict]:
    """
    Clean, type and enrich Bronze transactions.
    One Silver row must remain for every Bronze row.
    """
    silver = pd.DataFrame(
        {
            "transaction_id": clean_integer(bronze["id"]),
            "transaction_timestamp": pd.to_datetime(
                clean_text(bronze["date"]),
                errors="coerce",
                utc=True,
            ),
            "client_id": clean_integer(bronze["client_id"]),
            "card_id": clean_integer(bronze["card_id"]),
            "amount": clean_money(bronze["amount"]),
            "transaction_channel": clean_text(
                bronze["use_chip"]
            ),
            "merchant_id": clean_integer(
                bronze["merchant_id"]
            ),
            "merchant_city": clean_text(
                bronze["merchant_city"]
            ),
            "merchant_state": clean_text(
                bronze["merchant_state"]
            ),
            "merchant_zip": (
                clean_text(bronze["zip"])
                .str.replace(r"\.0$", "", regex=True)
            ),
            "mcc": clean_integer(bronze["mcc"]),
            "errors": (
                clean_text(bronze["errors"])
                .str.replace(
                    r"\s*,\s*",
                    ",",
                    regex=True,
                )
            ),
            "_source_file": bronze["_source_file"].astype(
                "string"
            ),
            "_ingested_at_utc": bronze[
                "_ingested_at_utc"
            ].astype("string"),
            "_bronze_row_num": bronze[
                "_bronze_row_num"
            ].astype("Int64"),
            "_pipeline_run_id": bronze[
                "_pipeline_run_id"
            ].astype("string"),
        }
    )

    silver["merchant_category"] = (
        silver["mcc"]
        .astype("string")
        .map(mcc_codes)
        .fillna("Unknown MCC")
        .astype("string")
    )

    # No fraud label means Unknown, not No.
    silver["fraud_label"] = (
        silver["transaction_id"]
        .astype("string")
        .map(fraud_map)
        .fillna("Unknown")
        .astype("string")
    )

    original_row_count = len(silver)

    silver = silver.merge(
        cards_safe,
        on="card_id",
        how="left",
        validate="many_to_one",
        sort=False,
    )

    silver = silver.merge(
        users_safe,
        on="client_id",
        how="left",
        validate="many_to_one",
        sort=False,
    )

    assert len(silver) == original_row_count

    mismatch_mask = (
        silver["card_owner_client_id"].notna()
        & silver["client_id"].notna()
        & silver["card_owner_client_id"].ne(
            silver["client_id"]
        )
    )

    join_stats = {
        "missing_card_matches": int(
            (~silver["_card_joined"].fillna(False)).sum()
        ),
        "missing_user_matches": int(
            (~silver["_user_joined"].fillna(False)).sum()
        ),
        "card_client_mismatches": int(
            mismatch_mask.fillna(False).sum()
        ),
    }

    silver = silver.drop(
        columns=[
            "card_owner_client_id",
            "_card_joined",
            "_user_joined",
        ]
    )

    for column in [
        "card_brand",
        "card_type",
    ]:
        silver[column] = silver[column].astype("string")

    silver = silver[SILVER_COLUMNS]

    assert len(silver) == len(bronze)

    return silver, join_stats


def build_gold(silver: pd.DataFrame) -> pd.DataFrame:
    """
    Produce one safe case per transaction containing an error.
    """
    gold = silver[
        silver["errors"].notna()
    ].copy()

    gold = gold.rename(
        columns={
            "errors": "error_types",
        }
    )

    gold["exception_id"] = (
        "EXC-"
        + gold["transaction_id"].astype("string")
    )

    gold["masked_client_id"] = mask_identifier(
        gold["client_id"],
        prefix="CLIENT",
    )

    gold["masked_card_id"] = mask_identifier(
        gold["card_id"],
        prefix="CARD",
    )

    gold["is_multi_error"] = (
        gold["error_types"]
        .str.contains(",", regex=False)
        .fillna(False)
        .astype("boolean")
    )

    gold = gold[GOLD_COLUMNS]

    return gold

In [7]:
SILVER_REQUIRED_NOT_NULL = [
    "transaction_id",
    "transaction_timestamp",
    "client_id",
    "card_id",
    "amount",
]

GOLD_REQUIRED_NOT_NULL = [
    "exception_id",
    "transaction_id",
    "transaction_timestamp",
    "amount",
    "error_types",
]


SILVER_KEY_TYPES = {
    "transaction_id": "integer",
    "transaction_timestamp": "datetime",
    "amount": "number",
    "errors": "text",
    "fraud_label": "text",
    "has_chip": "boolean",
}

GOLD_KEY_TYPES = {
    "exception_id": "text",
    "transaction_id": "integer",
    "transaction_timestamp": "datetime",
    "amount": "number",
    "error_types": "text",
    "is_multi_error": "boolean",
}


def type_family(series: pd.Series) -> str:
    if pd.api.types.is_bool_dtype(series):
        return "boolean"

    if pd.api.types.is_datetime64_any_dtype(series):
        return "datetime"

    if pd.api.types.is_integer_dtype(series):
        return "integer"

    if pd.api.types.is_numeric_dtype(series):
        return "number"

    return "text"


def test_schema(
    dataframe: pd.DataFrame,
    expected_columns: list[str],
    expected_types: dict[str, str],
) -> dict:
    actual_columns = list(dataframe.columns)

    missing_columns = [
        column
        for column in expected_columns
        if column not in actual_columns
    ]

    unexpected_columns = [
        column
        for column in actual_columns
        if column not in expected_columns
    ]

    wrong_types = []

    for column, expected_type in expected_types.items():
        if column not in dataframe.columns:
            continue

        actual_type = type_family(dataframe[column])

        if actual_type != expected_type:
            wrong_types.append(
                {
                    "column": column,
                    "expected": expected_type,
                    "actual": actual_type,
                }
            )

    return {
        "test": "schema",
        "passed": (
            not missing_columns
            and not unexpected_columns
            and actual_columns == expected_columns
            and not wrong_types
        ),
        "missing_columns": missing_columns,
        "unexpected_columns": unexpected_columns,
        "order_ok": actual_columns == expected_columns,
        "wrong_types": wrong_types,
    }


def test_not_null(
    dataframe: pd.DataFrame,
    columns: list[str],
) -> dict:
    null_counts = {}

    for column in columns:
        if column not in dataframe.columns:
            null_counts[column] = "COLUMN MISSING"
        else:
            null_counts[column] = int(
                dataframe[column].isna().sum()
            )

    return {
        "test": "not_null",
        "passed": all(
            value == 0
            for value in null_counts.values()
        ),
        "null_counts": null_counts,
        "rows_checked": len(dataframe),
    }


def test_unique(
    dataframe: pd.DataFrame,
    column: str,
) -> dict:
    if column not in dataframe.columns:
        return {
            "test": "unique",
            "passed": False,
            "column": column,
            "duplicate_count": "COLUMN MISSING",
        }

    duplicate_count = int(
        dataframe[column].duplicated().sum()
    )

    return {
        "test": "unique",
        "passed": duplicate_count == 0,
        "column": column,
        "duplicate_count": duplicate_count,
    }


def run_gold_tests(
    dataframe: pd.DataFrame,
) -> tuple[bool, list[dict]]:
    results = [
        test_schema(
            dataframe,
            GOLD_COLUMNS,
            GOLD_KEY_TYPES,
        ),
        test_not_null(
            dataframe,
            GOLD_REQUIRED_NOT_NULL,
        ),
        test_unique(
            dataframe,
            "transaction_id",
        ),
    ]

    for result in results:
        status = "PASS" if result["passed"] else "FAIL"
        print(f"{status} {result['test']} | {result}")

    return (
        all(result["passed"] for result in results),
        results,
    )

In [8]:
def write_parquet_chunk(
    dataframe: pd.DataFrame,
    output_path: Path,
    writer: pq.ParquetWriter | None,
) -> pq.ParquetWriter | None:
    if dataframe.empty:
        return writer

    arrow_table = pa.Table.from_pandas(
        dataframe,
        preserve_index=False,
    )

    if writer is None:
        writer = pq.ParquetWriter(
            output_path,
            arrow_table.schema,
            compression="snappy",
        )

    writer.write_table(arrow_table)

    return writer

In [9]:
metrics = {
    "source_rows": 0,
    "silver_rows": 0,
    "source_error_rows": 0,
    "gold_rows": 0,
    "missing_card_matches": 0,
    "missing_user_matches": 0,
    "card_client_mismatches": 0,
    "unknown_fraud_labels": 0,
    "fraud_yes_in_gold": 0,
    "unknown_mcc_rows": 0,
}

silver_writer = None
gold_writer = None

row_offset = 0

bronze_preview = None
silver_preview = None
gold_preview = None


try:
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_file:
        with zip_file.open(
            dataset_files["transactions"]
        ) as transaction_file:

            transaction_reader = pd.read_csv(
                transaction_file,
                dtype=str,
                keep_default_na=False,
                chunksize=CHUNK_SIZE,
            )

            for chunk_number, raw_chunk in enumerate(
                transaction_reader,
                start=1,
            ):
                bronze = build_bronze(
                    raw_chunk,
                    row_offset=row_offset,
                )

                silver, join_stats = build_silver(bronze)

                silver_schema_test = test_schema(
                    silver,
                    SILVER_COLUMNS,
                    SILVER_KEY_TYPES,
                )

                silver_null_test = test_not_null(
                    silver,
                    SILVER_REQUIRED_NOT_NULL,
                )

                silver_tests_passed = (
                    silver_schema_test["passed"]
                    and silver_null_test["passed"]
                )

                if not silver_tests_passed:
                    print("Silver schema test:", silver_schema_test)
                    print("Silver null test:", silver_null_test)

                    raise RuntimeError(
                        f"Silver quality gate failed "
                        f"for chunk {chunk_number}"
                    )

                # Gold is created only after Silver passes.
                gold = build_gold(silver)

                gold_schema_test = test_schema(
                    gold,
                    GOLD_COLUMNS,
                    GOLD_KEY_TYPES,
                )

                gold_null_test = test_not_null(
                    gold,
                    GOLD_REQUIRED_NOT_NULL,
                )

                if not (
                    gold_schema_test["passed"]
                    and gold_null_test["passed"]
                ):
                    print("Gold schema test:", gold_schema_test)
                    print("Gold null test:", gold_null_test)

                    raise RuntimeError(
                        f"Gold quality gate failed "
                        f"for chunk {chunk_number}"
                    )

                silver_writer = write_parquet_chunk(
                    silver,
                    SILVER_PATH,
                    silver_writer,
                )

                gold_writer = write_parquet_chunk(
                    gold,
                    GOLD_PATH,
                    gold_writer,
                )

                source_errors = clean_text(
                    bronze["errors"]
                ).notna()

                metrics["source_rows"] += len(bronze)
                metrics["silver_rows"] += len(silver)
                metrics["source_error_rows"] += int(
                    source_errors.sum()
                )
                metrics["gold_rows"] += len(gold)

                metrics["missing_card_matches"] += (
                    join_stats["missing_card_matches"]
                )

                metrics["missing_user_matches"] += (
                    join_stats["missing_user_matches"]
                )

                metrics["card_client_mismatches"] += (
                    join_stats["card_client_mismatches"]
                )

                metrics["unknown_fraud_labels"] += int(
                    silver["fraud_label"]
                    .eq("Unknown")
                    .sum()
                )

                metrics["fraud_yes_in_gold"] += int(
                    gold["fraud_label"]
                    .eq("Yes")
                    .sum()
                )

                metrics["unknown_mcc_rows"] += int(
                    silver["merchant_category"]
                    .eq("Unknown MCC")
                    .sum()
                )

                if chunk_number == 1:
                    bronze_preview = bronze.head(5).copy()
                    silver_preview = silver.head(5).copy()
                    gold_preview = gold.head(10).copy()

                row_offset += len(bronze)

                print(
                    f"Chunk {chunk_number:02d} | "
                    f"source={metrics['source_rows']:,} | "
                    f"silver={metrics['silver_rows']:,} | "
                    f"gold={metrics['gold_rows']:,}"
                )

                del raw_chunk
                del bronze
                del silver
                del gold

finally:
    if silver_writer is not None:
        silver_writer.close()

    if gold_writer is not None:
        gold_writer.close()


print("\nPipeline completed.")

Chunk 01 | source=500,000 | silver=500,000 | gold=7,660
Chunk 02 | source=1,000,000 | silver=1,000,000 | gold=15,623
Chunk 03 | source=1,500,000 | silver=1,500,000 | gold=23,499
Chunk 04 | source=2,000,000 | silver=2,000,000 | gold=31,484
Chunk 05 | source=2,500,000 | silver=2,500,000 | gold=39,481
Chunk 06 | source=3,000,000 | silver=3,000,000 | gold=47,339
Chunk 07 | source=3,500,000 | silver=3,500,000 | gold=55,225
Chunk 08 | source=4,000,000 | silver=4,000,000 | gold=63,094
Chunk 09 | source=4,500,000 | silver=4,500,000 | gold=71,085
Chunk 10 | source=5,000,000 | silver=5,000,000 | gold=79,083
Chunk 11 | source=5,500,000 | silver=5,500,000 | gold=87,082
Chunk 12 | source=6,000,000 | silver=6,000,000 | gold=94,810
Chunk 13 | source=6,500,000 | silver=6,500,000 | gold=102,780
Chunk 14 | source=7,000,000 | silver=7,000,000 | gold=110,832
Chunk 15 | source=7,500,000 | silver=7,500,000 | gold=118,890
Chunk 16 | source=8,000,000 | silver=8,000,000 | gold=126,877
Chunk 17 | source=8,500,0

In [10]:
print("BRONZE PREVIEW")
display(bronze_preview)

print("\nSILVER PREVIEW")
display(silver_preview)

print("\nGOLD PREVIEW")
display(gold_preview)

BRONZE PREVIEW


,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors,_source_file,_ingested_at_utc,_bronze_row_num,_pipeline_run_id
0,7475327,2010-01-01 00:01:00,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,,transactions_data.csv,2026-08-06T13:25:09.992818+00:00,1,20260806T132509Z
1,7475328,2010-01-01 00:02:00,561,4575,$14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,,transactions_data.csv,2026-08-06T13:25:09.992818+00:00,2,20260806T132509Z
2,7475329,2010-01-01 00:02:00,1129,102,$80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,,transactions_data.csv,2026-08-06T13:25:09.992818+00:00,3,20260806T132509Z
3,7475331,2010-01-01 00:05:00,430,2860,$200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,,transactions_data.csv,2026-08-06T13:25:09.992818+00:00,4,20260806T132509Z
4,7475332,2010-01-01 00:06:00,848,3915,$46.41,Swipe Transaction,13051,Harwood,MD,20776.0,5813,,transactions_data.csv,2026-08-06T13:25:09.992818+00:00,5,20260806T132509Z



SILVER PREVIEW


,transaction_id,transaction_timestamp,client_id,card_id,amount,transaction_channel,merchant_id,merchant_city,merchant_state,merchant_zip,...,card_on_dark_web,current_age,yearly_income,total_debt,credit_score,num_credit_cards,_source_file,_ingested_at_utc,_bronze_row_num,_pipeline_run_id
0,7475327,2010-01-01 00:01:00+00:00,1556,2972,-77.0,Swipe Transaction,59935,Beulah,ND,58523,...,False,30,48277.0,110153.0,740,4,transactions_data.csv,2026-08-06T13:25:09.992818+00:00,1,20260806T132509Z
1,7475328,2010-01-01 00:02:00+00:00,561,4575,14.57,Swipe Transaction,67570,Bettendorf,IA,52722,...,False,48,36853.0,112139.0,834,5,transactions_data.csv,2026-08-06T13:25:09.992818+00:00,2,20260806T132509Z
2,7475329,2010-01-01 00:02:00+00:00,1129,102,80.0,Swipe Transaction,27092,Vista,CA,92084,...,False,49,34449.0,36540.0,686,3,transactions_data.csv,2026-08-06T13:25:09.992818+00:00,3,20260806T132509Z
3,7475331,2010-01-01 00:05:00+00:00,430,2860,200.0,Swipe Transaction,27092,Crown Point,IN,46307,...,False,52,53350.0,128676.0,685,5,transactions_data.csv,2026-08-06T13:25:09.992818+00:00,4,20260806T132509Z
4,7475332,2010-01-01 00:06:00+00:00,848,3915,46.41,Swipe Transaction,13051,Harwood,MD,20776,...,False,51,68362.0,96182.0,711,2,transactions_data.csv,2026-08-06T13:25:09.992818+00:00,5,20260806T132509Z



GOLD PREVIEW


,exception_id,transaction_id,transaction_timestamp,masked_client_id,masked_card_id,amount,transaction_channel,merchant_id,merchant_city,merchant_state,...,card_on_dark_web,current_age,yearly_income,total_debt,credit_score,num_credit_cards,_source_file,_ingested_at_utc,_bronze_row_num,_pipeline_run_id
166,EXC-7475516,7475516,2010-01-01 04:56:00+00:00,CLIENT-88b54564b232,CARD-84d133d96852,104.1,Swipe Transaction,32175,Orlando,FL,...,False,36,67444.0,93513.0,850,1,transactions_data.csv,2026-08-06T13:25:09.992818+00:00,167,20260806T132509Z
248,EXC-7475611,7475611,2010-01-01 06:10:00+00:00,CLIENT-f8b4b02c09cf,CARD-7377a71607a8,28.84,Online Transaction,15143,ONLINE,<NA>,...,False,67,30962.0,15336.0,743,5,transactions_data.csv,2026-08-06T13:25:09.992818+00:00,249,20260806T132509Z
275,EXC-7475643,7475643,2010-01-01 06:19:00+00:00,CLIENT-f8b4b02c09cf,CARD-7377a71607a8,38.58,Online Transaction,15143,ONLINE,<NA>,...,False,67,30962.0,15336.0,743,5,transactions_data.csv,2026-08-06T13:25:09.992818+00:00,276,20260806T132509Z
401,EXC-7475792,7475792,2010-01-01 07:02:00+00:00,CLIENT-a478642504ac,CARD-6aa7d46a7422,-72.0,Swipe Transaction,59935,Kingman,AZ,...,False,101,15348.0,1396.0,761,4,transactions_data.csv,2026-08-06T13:25:09.992818+00:00,402,20260806T132509Z
483,EXC-7475881,7475881,2010-01-01 07:22:00+00:00,CLIENT-55e8ab098d48,CARD-52f11620e397,37.54,Swipe Transaction,89462,Terre Haute,IN,...,False,35,28325.0,38658.0,822,6,transactions_data.csv,2026-08-06T13:25:09.992818+00:00,484,20260806T132509Z
484,EXC-7475882,7475882,2010-01-01 07:22:00+00:00,CLIENT-a478642504ac,CARD-6aa7d46a7422,72.0,Swipe Transaction,59935,Kingman,AZ,...,False,101,15348.0,1396.0,761,4,transactions_data.csv,2026-08-06T13:25:09.992818+00:00,485,20260806T132509Z
524,EXC-7475935,7475935,2010-01-01 07:37:00+00:00,CLIENT-156091ee0884,CARD-766cb53c753b,104.81,Swipe Transaction,9263,Fresno,CA,...,False,67,28357.0,10400.0,628,2,transactions_data.csv,2026-08-06T13:25:09.992818+00:00,525,20260806T132509Z
577,EXC-7476004,7476004,2010-01-01 07:51:00+00:00,CLIENT-58eb0dd988df,CARD-50e65b145580,90.1,Online Transaction,38958,ONLINE,<NA>,...,False,49,44383.0,42369.0,707,7,transactions_data.csv,2026-08-06T13:25:09.992818+00:00,578,20260806T132509Z
715,EXC-7476174,7476174,2010-01-01 08:36:00+00:00,CLIENT-72b31cf00f8a,CARD-bcb2c2ec064c,2.46,Swipe Transaction,40948,Westminster,CA,...,False,59,39633.0,58586.0,777,3,transactions_data.csv,2026-08-06T13:25:09.992818+00:00,716,20260806T132509Z
1000,EXC-7476516,7476516,2010-01-01 09:58:00+00:00,CLIENT-e52d08747b9d,CARD-8202c37e994f,1.31,Swipe Transaction,69831,Grabill,IN,...,False,61,21397.0,54771.0,812,4,transactions_data.csv,2026-08-06T13:25:09.992818+00:00,1001,20260806T132509Z


In [11]:
silver_rows_on_disk = (
    pq.ParquetFile(SILVER_PATH)
    .metadata
    .num_rows
)

gold_rows_on_disk = (
    pq.ParquetFile(GOLD_PATH)
    .metadata
    .num_rows
)

# Gold is small enough to perform a complete uniqueness test.
gold_keys = pd.read_parquet(
    GOLD_PATH,
    columns=[
        "transaction_id",
        "exception_id",
        "error_types",
    ],
)

quality_results = [
    {
        "test": "bronze_to_silver_row_count",
        "passed": (
            metrics["source_rows"]
            == metrics["silver_rows"]
            == silver_rows_on_disk
        ),
        "source_rows": metrics["source_rows"],
        "silver_rows": metrics["silver_rows"],
        "silver_rows_on_disk": silver_rows_on_disk,
    },
    {
        "test": "exception_row_reconciliation",
        "passed": (
            metrics["source_error_rows"]
            == metrics["gold_rows"]
            == gold_rows_on_disk
        ),
        "source_error_rows": metrics["source_error_rows"],
        "gold_rows": metrics["gold_rows"],
        "gold_rows_on_disk": gold_rows_on_disk,
    },
    {
        "test": "transaction_to_card_join",
        "passed": metrics["missing_card_matches"] == 0,
        "missing_matches": metrics["missing_card_matches"],
    },
    {
        "test": "transaction_to_user_join",
        "passed": metrics["missing_user_matches"] == 0,
        "missing_matches": metrics["missing_user_matches"],
    },
    {
        "test": "card_owner_consistency",
        "passed": metrics["card_client_mismatches"] == 0,
        "mismatches": metrics["card_client_mismatches"],
    },
    {
        "test": "gold_transaction_id_unique",
        "passed": not gold_keys[
            "transaction_id"
        ].duplicated().any(),
        "duplicate_count": int(
            gold_keys["transaction_id"]
            .duplicated()
            .sum()
        ),
    },
    {
        "test": "gold_exception_id_unique",
        "passed": not gold_keys[
            "exception_id"
        ].duplicated().any(),
        "duplicate_count": int(
            gold_keys["exception_id"]
            .duplicated()
            .sum()
        ),
    },
    {
        "test": "gold_critical_fields_not_null",
        "passed": (
            gold_keys[
                [
                    "transaction_id",
                    "exception_id",
                    "error_types",
                ]
            ]
            .isna()
            .sum()
            .sum()
            == 0
        ),
        "null_count": int(
            gold_keys[
                [
                    "transaction_id",
                    "exception_id",
                    "error_types",
                ]
            ]
            .isna()
            .sum()
            .sum()
        ),
    },
]

for result in quality_results:
    status = "PASS" if result["passed"] else "FAIL"
    print(f"{status} {result['test']} | {result}")

all_tests_passed = all(
    result["passed"]
    for result in quality_results
)

print("\nAll final tests passed:", all_tests_passed)

PASS bronze_to_silver_row_count | {'test': 'bronze_to_silver_row_count', 'passed': True, 'source_rows': 13305915, 'silver_rows': 13305915, 'silver_rows_on_disk': 13305915}
PASS exception_row_reconciliation | {'test': 'exception_row_reconciliation', 'passed': True, 'source_error_rows': 211393, 'gold_rows': 211393, 'gold_rows_on_disk': 211393}
PASS transaction_to_card_join | {'test': 'transaction_to_card_join', 'passed': True, 'missing_matches': 0}
PASS transaction_to_user_join | {'test': 'transaction_to_user_join', 'passed': True, 'missing_matches': 0}
PASS card_owner_consistency | {'test': 'card_owner_consistency', 'passed': True, 'mismatches': 0}
PASS gold_transaction_id_unique | {'test': 'gold_transaction_id_unique', 'passed': True, 'duplicate_count': 0}
PASS gold_exception_id_unique | {'test': 'gold_exception_id_unique', 'passed': True, 'duplicate_count': 0}
PASS gold_critical_fields_not_null | {'test': 'gold_critical_fields_not_null', 'passed': True, 'null_count': 0}

All final tes

In [12]:
quality_evidence = {
    "pipeline_run_id": RUN_ID,
    "source_zip": ZIP_PATH.name,
    "silver_output": str(SILVER_PATH),
    "gold_output": str(GOLD_PATH),
    "metrics": metrics,
    "quality_results": quality_results,
    "all_tests_passed": all_tests_passed,
}

with QUALITY_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        quality_evidence,
        file,
        indent=2,
        default=str,
    )

print("Saved Silver:", SILVER_PATH)
print("Saved Gold:", GOLD_PATH)
print("Saved quality evidence:", QUALITY_PATH)

Saved Silver: /home/labuser/Desktop/Persistent_Folder/Capstone/Resolve1/data/processed/silver_transactions.parquet
Saved Gold: /home/labuser/Desktop/Persistent_Folder/Capstone/Resolve1/data/processed/gold_exception_cases.parquet
Saved quality evidence: /home/labuser/Desktop/Persistent_Folder/Capstone/Resolve1/data/processed/quality_results.json


In [13]:
broken_gold = gold_preview.copy()

# Simulate schema drift
broken_gold = broken_gold.rename(
    columns={
        "error_types": "error_type",
    }
)

# Simulate quality drift
broken_gold.loc[
    broken_gold.index[:2],
    "transaction_id",
] = pd.NA

print("--- expecting tests to fail ---")

broken_passed, broken_results = run_gold_tests(
    broken_gold
)

print(
    "\nAll negative-control tests passed:",
    broken_passed,
)

print(
    "Expected result: False"
)

--- expecting tests to fail ---
FAIL schema | {'test': 'schema', 'passed': False, 'missing_columns': ['error_types'], 'unexpected_columns': ['error_type'], 'order_ok': False, 'wrong_types': []}
FAIL not_null | {'test': 'not_null', 'passed': False, 'null_counts': {'exception_id': 0, 'transaction_id': 2, 'transaction_timestamp': 0, 'amount': 0, 'error_types': 'COLUMN MISSING'}, 'rows_checked': 10}
FAIL unique | {'test': 'unique', 'passed': False, 'column': 'transaction_id', 'duplicate_count': 1}

All negative-control tests passed: False
Expected result: False


In [14]:
print("RESOLVEONE PIPELINE SUMMARY")
print("-" * 45)

print(
    "Source transactions:",
    f"{metrics['source_rows']:,}",
)

print(
    "Silver transactions:",
    f"{metrics['silver_rows']:,}",
)

print(
    "Exception cases:",
    f"{metrics['gold_rows']:,}",
)

print(
    "Unknown fraud labels:",
    f"{metrics['unknown_fraud_labels']:,}",
)

print(
    "Fraud-positive exception cases:",
    f"{metrics['fraud_yes_in_gold']:,}",
)

print(
    "Unmapped MCC rows:",
    f"{metrics['unknown_mcc_rows']:,}",
)

print(
    "All quality tests passed:",
    all_tests_passed,
)

print("\nGold grain:")
print("One row per transaction containing one or more errors.")

RESOLVEONE PIPELINE SUMMARY
---------------------------------------------
Source transactions: 13,305,915
Silver transactions: 13,305,915
Exception cases: 211,393
Unknown fraud labels: 4,390,952
Fraud-positive exception cases: 569
Unmapped MCC rows: 0
All quality tests passed: True

Gold grain:
One row per transaction containing one or more errors.


In [17]:
accepted_fraud_labels = {"Yes", "No", "Unknown"}

actual_fraud_labels = set(
    pd.read_parquet(
        GOLD_PATH,
        columns=["fraud_label"],
    )["fraud_label"].dropna().unique()
)

quality_results.append(
    {
        "test": "fraud_label_accepted_values",
        "passed": actual_fraud_labels.issubset(
            accepted_fraud_labels
        ),
        "unexpected_values": sorted(
            actual_fraud_labels - accepted_fraud_labels
        ),
    }
)
gold_errors = pd.read_parquet(
    GOLD_PATH,
    columns=["error_types"],
)["error_types"].astype("string")

empty_error_count = int(
    gold_errors.isna().sum()
    + gold_errors.str.strip().eq("").sum()
)

quality_results.append(
    {
        "test": "gold_errors_not_empty",
        "passed": empty_error_count == 0,
        "empty_error_count": empty_error_count,
    }
)
prohibited_columns = {
    "card_number",
    "cvv",
    "address",
    "latitude",
    "longitude",
    "client_id",
    "card_id",
}

gold_columns_on_disk = set(
    pq.ParquetFile(GOLD_PATH).schema.names
)

exposed_prohibited_columns = sorted(
    gold_columns_on_disk.intersection(prohibited_columns)
)

quality_results.append(
    {
        "test": "no_prohibited_columns_in_gold",
        "passed": not exposed_prohibited_columns,
        "exposed_columns": exposed_prohibited_columns,
    }
)
all_tests_passed = all(
    result["passed"]
    for result in quality_results
)

In [18]:
from pathlib import Path
import json

import pandas as pd
import pyarrow.parquet as pq


CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

GOLD_PATH = (
    PROCESSED_DIR
    / "gold_exception_cases.parquet"
)

QUALITY_PATH = (
    PROCESSED_DIR
    / "quality_results.json"
)


# Load the quality evidence already produced by the pipeline.
with QUALITY_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    quality_evidence = json.load(file)

quality_results = quality_evidence["quality_results"]


# Convert values such as "True" into real Python booleans.
for result in quality_results:
    value = result.get("passed", False)

    if isinstance(value, str):
        result["passed"] = value.strip().lower() == "true"
    else:
        result["passed"] = bool(value)


# Test 1: fraud labels contain only accepted values.
accepted_fraud_labels = {
    "Yes",
    "No",
    "Unknown",
}

actual_fraud_labels = set(
    pd.read_parquet(
        GOLD_PATH,
        columns=["fraud_label"],
    )["fraud_label"]
    .dropna()
    .unique()
)

unexpected_fraud_labels = sorted(
    actual_fraud_labels
    - accepted_fraud_labels
)


# Test 2: Gold contains no empty error descriptions.
gold_errors = pd.read_parquet(
    GOLD_PATH,
    columns=["error_types"],
)["error_types"].astype("string")

empty_error_count = int(
    gold_errors.isna().sum()
    + gold_errors
    .str.strip()
    .eq("")
    .fillna(False)
    .sum()
)


# Test 3: Gold contains no prohibited columns.
prohibited_columns = {
    "card_number",
    "cvv",
    "address",
    "latitude",
    "longitude",
    "client_id",
    "card_id",
}

gold_columns = set(
    pq.ParquetFile(GOLD_PATH).schema.names
)

exposed_prohibited_columns = sorted(
    gold_columns.intersection(
        prohibited_columns
    )
)


additional_results = [
    {
        "test": "fraud_label_accepted_values",
        "passed": not unexpected_fraud_labels,
        "unexpected_values": unexpected_fraud_labels,
    },
    {
        "test": "gold_errors_not_empty",
        "passed": empty_error_count == 0,
        "empty_error_count": empty_error_count,
    },
    {
        "test": "no_prohibited_columns_in_gold",
        "passed": not exposed_prohibited_columns,
        "exposed_columns": exposed_prohibited_columns,
    },
]


# Replace an existing test with the same name instead of duplicating it.
results_by_name = {
    result["test"]: result
    for result in quality_results
}

for result in additional_results:
    results_by_name[result["test"]] = result

quality_results = list(
    results_by_name.values()
)

all_tests_passed = all(
    bool(result["passed"])
    for result in quality_results
)


# Update and overwrite the quality evidence file.
quality_evidence["quality_results"] = quality_results
quality_evidence["all_tests_passed"] = all_tests_passed

with QUALITY_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        quality_evidence,
        file,
        indent=2,
    )


for result in quality_results:
    status = (
        "PASS"
        if result["passed"]
        else "FAIL"
    )

    print(
        f"{status} {result['test']} | {result}"
    )

print(
    "\nAll quality tests passed:",
    all_tests_passed,
)

print(
    "Updated:",
    QUALITY_PATH,
)

PASS bronze_to_silver_row_count | {'test': 'bronze_to_silver_row_count', 'passed': True, 'source_rows': 13305915, 'silver_rows': 13305915, 'silver_rows_on_disk': 13305915}
PASS exception_row_reconciliation | {'test': 'exception_row_reconciliation', 'passed': True, 'source_error_rows': 211393, 'gold_rows': 211393, 'gold_rows_on_disk': 211393}
PASS transaction_to_card_join | {'test': 'transaction_to_card_join', 'passed': True, 'missing_matches': 0}
PASS transaction_to_user_join | {'test': 'transaction_to_user_join', 'passed': True, 'missing_matches': 0}
PASS card_owner_consistency | {'test': 'card_owner_consistency', 'passed': True, 'mismatches': 0}
PASS gold_transaction_id_unique | {'test': 'gold_transaction_id_unique', 'passed': True, 'duplicate_count': 0}
PASS gold_exception_id_unique | {'test': 'gold_exception_id_unique', 'passed': True, 'duplicate_count': 0}
PASS gold_critical_fields_not_null | {'test': 'gold_critical_fields_not_null', 'passed': True, 'null_count': 0}
PASS fraud_lab